In [13]:
import pandas as pd
import numpy as np

TARGETS = ["x"]

In [15]:
results_1l = pd.read_excel("resultados-1l.xlsx")
# results_2l = pd.read_excel("resultados-2l.xlsx")
# results_3l = pd.read_excel("resultados-3l.xlsx")

# results = pd.concat(
#     [results_1l, results_2l,],
#     ignore_index=True )
results = results_1l


In [16]:
results

,model,Neurons,Ld,Lp,reg,seed,R2_ZZx1_x,MSE_ZZx1_x,R2_ZZx2_x,MSE_ZZx2_x,R2_ZZxReto_x,MSE_ZZxReto_x
0,model_arch79_r0.01_Ld0.5_Lp0.5_seed8998,[79],0.5,0.5,0.01,8998,0.993246,0.976849,-2.948644,0.817606,0.368659,0.943015
1,model_arch79_r0.01_Ld0.5_Lp0.5_seed569,[79],0.5,0.5,0.01,569,0.994780,0.977246,-4.400400,0.807704,0.322692,0.935777
2,model_arch79_r0.01_Ld0.5_Lp0.5_seed6066,[79],0.5,0.5,0.01,6066,0.883170,0.967355,0.799373,0.850929,0.861826,0.942107
3,model_arch79_r0.01_Ld0.5_Lp0.5_seed5663,[79],0.5,0.5,0.01,5663,0.815708,0.959558,0.854041,0.831576,0.922773,0.937783
4,model_arch79_r0.01_Ld0.5_Lp0.5_seed1791,[79],0.5,0.5,0.01,1791,0.988194,0.975254,-3.334605,0.799645,0.293236,0.943356
5,model_arch79_r0.9_Ld0.5_Lp0.5_seed8998,[79],0.5,0.5,0.90,8998,0.987898,0.976967,-3.064464,0.806967,0.484294,0.942103
6,model_arch79_r0.9_Ld0.5_Lp0.5_seed569,[79],0.5,0.5,0.90,569,0.905668,0.962697,-0.673922,0.856357,0.766484,0.929743
7,model_arch79_r0.9_Ld0.5_Lp0.5_seed6066,[79],0.5,0.5,0.90,6066,0.890384,0.961838,0.370693,0.862199,0.845876,0.935157
8,model_arch79_r0.9_Ld0.5_Lp0.5_seed5663,[79],0.5,0.5,0.90,5663,0.834398,0.958657,0.179401,0.849530,0.885044,0.930921
9,model_arch79_r0.9_Ld0.5_Lp0.5_seed1791,[79],0.5,0.5,0.90,1791,0.990981,0.976001,-2.898687,0.811634,0.381954,0.943795


In [17]:
# 🔹 categorização dos sets (baseada nos comentários originais)
SETS_CATEGORY = {
    "ZZx1":     "Train",
    "ZZx2":     "Val",
    "ZZxReto":  "Test",
}

def col_name(s, target):
    # sanitiza "-" pra "_" pra bater com o nome real da coluna, se for o caso
    return f"R2_{s.replace('-', '_')}_{target}"

best_models_tables = {}
N = 5  # top modelos

w_val = 0.33
w_train = 0.33
w_test = 0.33


for target in TARGETS:

    # 🔹 sets de Train, Val e Test
    train_sets = [s for s, cat in SETS_CATEGORY.items() if cat == "Train"]
    val_sets   = [s for s, cat in SETS_CATEGORY.items() if cat == "Val"]
    test_sets  = [s for s, cat in SETS_CATEGORY.items() if cat == "Test"]

    train_cols = [col_name(s, target) for s in train_sets]
    val_cols   = [col_name(s, target) for s in val_sets]
    test_cols  = [col_name(s, target) for s in test_sets]

    # 🔹 garantir que só usamos colunas existentes
    train_cols = [c for c in train_cols if c in results.columns]
    val_cols   = [c for c in val_cols if c in results.columns]
    test_cols  = [c for c in test_cols if c in results.columns]

    r2_all_cols = train_cols + val_cols + test_cols

    if not r2_all_cols:
        print(f"⚠️ Nenhuma coluna Train/Val/Test encontrada para target={target}, pulando.")
        continue

    df = results.copy()

    # 🔹 remover linhas onde QUALQUER R2 (Train/Val/Test) < 0
    # df = df[(df[r2_all_cols] >= 0).all(axis=1)]

    # =========================
    # 🔹 MÉDIAS POR GRUPO
    # =========================
    df["R2_train_mean"] = df[train_cols].mean(axis=1) if train_cols else np.nan
    df["R2_val_mean"]   = df[val_cols].mean(axis=1) if val_cols else np.nan
    df["R2_test_mean"]  = df[test_cols].mean(axis=1) if test_cols else np.nan

    # =========================
    # 🔹 SCORE
    # =========================
    df["R2_std"] = df[r2_all_cols].std(axis=1)

    df["Score"] = (
        w_train * df["R2_train_mean"] +
        w_val   * df["R2_val_mean"] +
        w_test  * df["R2_test_mean"] - 
        0.1 * df["R2_std"]   # penaliza inconsistência
    )

    # =========================
    # 🔹 ORDENAÇÃO
    # =========================
    df_sorted = df.sort_values(by="Score", ascending=False)
    best_models_tables[target] = df_sorted

    # =========================
    # 🔹 TOP N RESUMO
    # =========================
    print(f"\n🏆 TOP {N} MODELOS - {target}")
    display(df_sorted[
        ["model", "Neurons", "R2_train_mean", "R2_val_mean", "R2_test_mean", "Score"]
    ].head(N))

    top_df = df_sorted.head(N).copy()

    final_cols = ["model", "Neurons"] + r2_all_cols + [
        "R2_train_mean", "R2_val_mean", "R2_test_mean", "Score"
    ]
    final_table = top_df[final_cols]

    print(f"\n📊 MÉTRICAS COMPLETAS - TOP {N} ({target})")
    display(final_table)


🏆 TOP 5 MODELOS - x


,model,Neurons,R2_train_mean,R2_val_mean,R2_test_mean,Score
17,model_arch79_r0.9_Ld0.3_Lp0.7_seed6066,[79],0.885141,0.893251,0.885771,0.878723
15,model_arch79_r0.9_Ld0.3_Lp0.7_seed8998,[79],0.876991,0.814694,0.907878,0.853109
3,model_arch79_r0.01_Ld0.5_Lp0.5_seed5663,[79],0.815708,0.854041,0.922773,0.850108
10,model_arch79_r0.01_Ld0.3_Lp0.7_seed8998,[79],0.837138,0.840214,0.870522,0.838953
2,model_arch79_r0.01_Ld0.5_Lp0.5_seed6066,[79],0.883170,0.799373,0.861826,0.835287



📊 MÉTRICAS COMPLETAS - TOP 5 (x)


,model,Neurons,R2_ZZx1_x,R2_ZZx2_x,R2_ZZxReto_x,R2_train_mean,R2_val_mean,R2_test_mean,Score
17,model_arch79_r0.9_Ld0.3_Lp0.7_seed6066,[79],0.885141,0.893251,0.885771,0.885141,0.893251,0.885771,0.878723
15,model_arch79_r0.9_Ld0.3_Lp0.7_seed8998,[79],0.876991,0.814694,0.907878,0.876991,0.814694,0.907878,0.853109
3,model_arch79_r0.01_Ld0.5_Lp0.5_seed5663,[79],0.815708,0.854041,0.922773,0.815708,0.854041,0.922773,0.850108
10,model_arch79_r0.01_Ld0.3_Lp0.7_seed8998,[79],0.837138,0.840214,0.870522,0.837138,0.840214,0.870522,0.838953
2,model_arch79_r0.01_Ld0.5_Lp0.5_seed6066,[79],0.883170,0.799373,0.861826,0.883170,0.799373,0.861826,0.835287


In [18]:
final_table.to_excel("BestModels-otm.xlsx")